In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
from PIL import ImageFile

# Prevent crashes from slightly corrupted dataset images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# EfficientNetB3 optimal resolution is 300x300
IMG_SIZE = 300 
BATCH_SIZE = 32 # Keep at 32, drop to 16 if CUDA Out of Memory error occurs
DATA_PATH = '/kaggle/input/datasets/shruthisindhura/pestopia/Datasets/Pest_Dataset'

# ==========================================
# 2. FOCAL LOSS IMPLEMENTATION
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

# ==========================================
# 3. ADVANCED AUGMENTATION & DATA LOADING
# ==========================================
def get_dataloaders():
    train_tfms = transforms.Compose([
        transforms.Resize((320, 320)),
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30), 
        transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.2, scale=(0.8, 1.2)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    full_ds = datasets.ImageFolder(DATA_PATH)
    train_sz = int(0.8 * len(full_ds))
    val_sz = len(full_ds) - train_sz
    
    generator = torch.Generator().manual_seed(42)
    train_idx, val_idx = random_split(range(len(full_ds)), [train_sz, val_sz], generator=generator)
    
    train_ds = datasets.ImageFolder(DATA_PATH, transform=train_tfms)
    val_ds = datasets.ImageFolder(DATA_PATH, transform=val_tfms)
    
    train_sub = torch.utils.data.Subset(train_ds, train_idx.indices)
    val_sub = torch.utils.data.Subset(val_ds, val_idx.indices)
    
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, len(full_ds.classes)

train_loader, val_loader, NUM_CLASSES = get_dataloaders()
print(f"✅ Data Setup Complete. Classes: {NUM_CLASSES}")

# ==========================================
# 4. EFFICIENTNET-B3 MODEL SETUP
# ==========================================
model = models.efficientnet_b3(weights='DEFAULT')

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    # FIX 1: Increased Dropout from 0.4 to 0.5 to prevent memorization
    nn.Dropout(p=0.5, inplace=True),
    nn.Linear(num_ftrs, NUM_CLASSES)
)
model = model.to(DEVICE)

# ==========================================
# 5. OPTIMIZER & SCHEDULER SETUP
# ==========================================
criterion = FocalLoss(gamma=2.0)

# FIX 2: Switched to AdamW and added heavy Weight Decay (1e-3)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)

# Scheduler (Removed 'verbose=True' to prevent Kaggle crash)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)

def run_epoch(loader, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct = 0.0, 0
    iterator = tqdm(loader, desc="Training", leave=False) if is_train else loader
    
    with torch.set_grad_enabled(is_train):
        for img, label in iterator:
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            if is_train:
                optimizer.zero_grad()
                out = model(img)
                loss = criterion(out, label)
                loss.backward()
                optimizer.step()
            else:
                out = model(img)
                loss = criterion(out, label)
            
            total_loss += loss.item() * img.size(0)
            correct += (out.argmax(1) == label).sum().item()
            
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ==========================================
# 6. TRAINING LOOP WITH EARLY STOPPING
# ==========================================
EPOCHS = 40
best_acc = 0.0
patience, patience_counter = 7, 0 

print(f"\n🚀 Starting Training for up to {EPOCHS} Epochs...")
for epoch in range(EPOCHS):
    t_loss, t_acc = run_epoch(train_loader, True)
    v_loss, v_acc = run_epoch(val_loader, False)
    
    scheduler.step(v_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Ep {epoch+1:02d}/{EPOCHS} | LR: {current_lr:.6f} | Train Acc: {t_acc*100:.2f}% | Val Acc: {v_acc*100:.2f}% | Val Loss: {v_loss:.4f}")
    
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'efficientnet_b3_pestopia.pth')
        print(f"   🏆 New Best Validation Score Saved! ({best_acc*100:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"\n🛑 Early stopping triggered. Validation accuracy plateaued for {patience} epochs.")
        break

print(f"\n✅ Training Complete. Best Accuracy Achieved: {best_acc*100:.2f}%")

Using device: cuda
✅ Data Setup Complete. Classes: 132
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 212MB/s]



🚀 Starting Training for up to 40 Epochs...


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Ep 01/40 | LR: 0.001000 | Train Acc: 39.40% | Val Acc: 55.75% | Val Loss: 1.3879
   🏆 New Best Validation Score Saved! (55.75%)


...

Ep 33/40 | LR: 0.000031 | Train Acc: 93.91% | Val Acc: 73.21% | Val Loss: 1.0459


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Ep 34/40 | LR: 0.000016 | Train Acc: 93.93% | Val Acc: 73.21% | Val Loss: 1.0400


Training:   0%|          | 0/1398 [00:00<?, ?it/s]

Ep 35/40 | LR: 0.000016 | Train Acc: 94.02% | Val Acc: 73.40% | Val Loss: 1.0495

🛑 Early stopping triggered. Validation accuracy plateaued for 7 epochs.

✅ Training Complete. Best Accuracy Achieved: 73.42%
